# 06. Context Management (컨텍스트 관리)

에이전트 실행 중 **사용자 정보(user id, 권한 등), DB 연결 객체, API 키, 설정값** 등을 도구 함수에 전달해야 할 때가 있습니다.
`RunContextWrapper`를 사용하면 이러한 데이터를 **의존성 주입(Dependency Injection)** 방식으로 전달할 수 있습니다.

### 핵심 개념

| 개념 | 설명 |
|------|------|
| **Local Context** | 도구 함수에서 접근 가능한 로컬 데이터 (LLM에 전송되지 않음) |
| **RunContextWrapper[T]** | 컨텍스트를 감싸는 래퍼 클래스 |
| **ToolContext[T]** | RunContextWrapper + 도구 메타데이터 (도구 이름, 호출 ID 등) |

### 중요: 컨텍스트는 LLM에 전송되지 않습니다!
컨텍스트 객체는 **로컬에서만** 사용됩니다. 보안이 필요한 정보(API 키, DB 연결 등)를
안전하게 도구에 전달할 수 있습니다.

## 1. 기본 컨텍스트 사용법

`@dataclass`로 컨텍스트 객체를 정의하고, 도구 함수에서 `RunContextWrapper`를 통해 접근합니다.

In [ ]:
# ============================================================
# 컨텍스트 객체 정의
# - 에이전트 실행 전체에서 공유될 사용자 정보를 담는 데이터 클래스
# - @dataclass 데코레이터로 __init__, __repr__ 등을 자동 생성
# ============================================================
class UserContext:
# ============================================================
# 도구 함수 1: 사용자 프로필 조회
# - @function_tool: LLM이 호출할 수 있는 도구로 등록
# - RunContextWrapper[UserContext]: 타입 힌트로 컨텍스트 타입을 명시
#   → 실제 컨텍스트 객체는 wrapper.context로 접근
# ============================================================
    # RunContextWrapper에서 실제 UserContext 객체 추출
    # is_premium 값에 따라 멤버십 등급 문자열 결정
    # 사용자 정보를 포맷팅하여 반환 (LLM이 응답에 활용)
# ============================================================
# 도구 함수 2: 개인화 추천 항목 반환
# - 컨텍스트의 is_premium 값에 따라 다른 추천 콘텐츠 제공
# - LLM에게 파라미터를 요구하지 않고 컨텍스트에서 직접 정보를 가져옴
#   → 민감한 사용자 정보가 LLM 프롬프트에 노출되지 않는 보안상 이점
# ============================================================
    # RunContextWrapper에서 실제 UserContext 객체 추출
    # 멤버십 등급에 따라 차별화된 추천 콘텐츠 반환
        # 프리미엄 사용자: 고급 콘텐츠 및 1:1 서비스 추천
        # 일반 사용자: 입문 콘텐츠 및 커뮤니티 기반 서비스 추천

## 2. 에이전트 생성 및 실행

`Agent[T]`로 컨텍스트 타입을 지정하고, `Runner.run()`의 `context` 파라미터로 전달합니다.

In [ ]:
# 컨텍스트 타입이 지정된 에이전트
# 프리미엄 사용자 컨텍스트
# 컨텍스트를 전달하며 에이전트 실행

## 3. 동일한 에이전트, 다른 컨텍스트

같은 에이전트를 **다른 사용자 컨텍스트**로 실행하면 개인화된 결과를 얻습니다.

In [ ]:
# 일반 사용자 컨텍스트

## 4. 실전 예제: 컨텍스트에 서비스 의존성 주입

실무에서는 DB 연결, API 클라이언트 등의 **서비스 의존성**을 컨텍스트에 담아 전달합니다.

In [ ]:
class AppContext:
# 주문 조회 도구 - 컨텍스트의 DB를 사용
# 에이전트 생성
# 애플리케이션 컨텍스트 생성 (DB, 설정 등 포함)

## 5. ToolContext로 도구 메타데이터 접근

`ToolContext`는 `RunContextWrapper`를 확장하여 **도구 자체의 메타데이터**에도 접근할 수 있습니다.
- `ctx.tool_name`: 호출된 도구 이름
- `ctx.tool_call_id`: 도구 호출 고유 ID
- `ctx.tool_arguments`: 원본 인자 문자열

In [ ]:
# ============================================================
# ToolContext = RunContextWrapper를 상속한 클래스
# RunContextWrapper 기본 기능 + 도구 실행 메타데이터 추가 제공
# ============================================================
    # ToolContext 전용 메타데이터 출력 (RunContextWrapper에는 없는 정보)
    # RunContextWrapper와 동일하게 ctx.context로 UserContext 접근
# ============================================================
# Agent[UserContext]: 이 에이전트가 UserContext 타입을 사용함을 명시
# → 등록된 도구들도 동일한 UserContext 타입을 사용해야 함
# ============================================================
# ============================================================
# Runner.run(): 에이전트 실행
#     → 내부적으로 RunContextWrapper/ToolContext에 자동으로 래핑되어 전달됨
# ============================================================
# result.final_output: 에이전트가 최종적으로 반환한 텍스트 응답

### 정리

| 개념 | 코드 | 설명 |
|------|------|------|
| 컨텍스트 정의 | `@dataclass class MyContext` | 전달할 데이터/의존성 정의 |
| 에이전트 타입 | `Agent[MyContext]` | 컨텍스트 타입 지정 |
| 도구에서 접근 | `wrapper: RunContextWrapper[MyContext]` | 도구 함수의 첫 번째 인자 |
| 실행 시 전달 | `Runner.run(..., context=my_ctx)` | context 파라미터로 전달 |
| 도구 메타데이터 | `ctx: ToolContext[MyContext]` | 도구 이름, 호출 ID 등 추가 접근 |

**활용 사례:**
- 사용자별 개인화 (프로필, 권한, 설정)
- DB 연결/API 클라이언트 주입
- 로깅/감사 추적 (audit trail)
- 요청별 설정값 전달